Want to create a file with every records of tvtropes.clusters.txt and add the gender and age of the actor (from character.metadata.tsv)

In [7]:
import pandas as pd
import os
import json


In [6]:
# 1. Define Paths
DATA_PATH = 'data/MovieSummaries/'
OUTPUT_DIR = 'data/processed'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 2. Load and Parse Tropes
print("Loading tropes...")
trope_path = os.path.join(DATA_PATH, 'tvtropes.clusters.txt')
trope_df = pd.read_csv(trope_path, sep='\t', names=['trope_name', 'char_details'], header=None)

def parse_char_id(x):
    try:
        return json.loads(x).get('id')
    except:
        return None

trope_df['freebase_char_id'] = trope_df['char_details'].apply(parse_char_id)

# 3. Load Character Metadata
print("Loading character metadata...")
char_path = os.path.join(DATA_PATH, 'character.metadata.tsv')
char_cols = [
    'wiki_id', 'fb_id', 'rel_date', 'char_name', 'dob', 'gender_id', 
    'height', 'ethnicity', 'actor_name', 'age', 'freebase_char_id', 'fb_act_id', 'fb_id_2'
]
chars = pd.read_csv(char_path, sep='\t', names=char_cols, header=None, low_memory=False)

# Mappatura flessibile
def map_gender(g):
    if g == '/m/05zppz' or g == 'M':
        return 'M'
    elif g == '/m/02zsn' or g == 'F':
        return 'F'
    else:
        return None

# Applica la funzione alla colonna del genere
chars['gender'] = chars['gender_id'].apply(map_gender)

# 4. Merge Data
print("Merging datasets...")
# We use a left join to keep all trope records
final_df = pd.merge(
    trope_df[['trope_name', 'freebase_char_id']], 
    chars[['freebase_char_id', 'char_name', 'gender', 'actor_name', 'age']], 
    on='freebase_char_id', 
    how='left'
)

# 5. Save the result
output_file = os.path.join(OUTPUT_DIR, 'at_tropes_with_gender.csv')
final_df.to_csv(output_file, index=False)

print(f"Success! Saved {len(final_df)} records to {output_file}")

Loading tropes...
Loading character metadata...
Merging datasets...
Success! Saved 501 records to data/processed/at_tropes_with_gender.csv


now im joing the character file with the movie file to add movie name at the character file

In [8]:
char_cols = [
    'wiki_id', 
    'freebase_movie_id', 
    'release_date', 
    'char_name', 
    'actor_dob', 
    'actor_gender', 
    'actor_height', 
    'actor_ethnicity', 
    'actor_name', 
    'actor_age', 
    'freebase_char_id', 
    'freebase_actor_id', 
    'freebase_id_2'
]

# Caricamento corretto
df_char = pd.read_csv(
    "data/MovieSummaries/character.metadata.tsv", 
    sep='\t',          # Fondamentale per i file .tsv
    header=None,       # Dice a pandas che non ci sono titoli nella prima riga
    names=char_cols    # Assegna i nomi che abbiamo definito sopra
)

In [9]:
movie_cols = ['wiki_id', 'freebase_id', 'name', 'release_date', 'revenue', 'runtime', 'languages', 'countries', 'genres'] # poi sotto li assegno ai pezzetti
df_movie = pd.read_csv(os.path.join(DATA_PATH, 'movie.metadata.tsv'), sep='\t', names=movie_cols, header=None)

In [10]:
import os
import pandas as pd


movie_subset = df_movie[['freebase_id', 'name']]

# 2. merging
print("Merging characters with movie names...")
final_df = pd.merge(
    df_char, 
    movie_subset, 
    left_on='freebase_movie_id', 
    right_on='freebase_id', 
    how='left'
)

# Rinominiamo 'name' in 'movie_name' per non confonderlo 
final_df = final_df.rename(columns={'name': 'movie_name'})

# Rimuoviamo la colonna duplicata dell'ID (freebase_id è uguale a freebase_movie_id)
final_df = final_df.drop(columns=['freebase_id'])

# 4. Salvataggio
OUTPUT_DIR = 'data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_file = os.path.join(OUTPUT_DIR, 'char_and_movie.csv')

final_df.to_csv(output_file, index=False)

print(f"Successo! File salvato in: {output_file}")
print(f"Colonne finali: {final_df.columns.tolist()}")

Merging characters with movie names...
Successo! File salvato in: data/processed/char_and_movie.csv
Colonne finali: ['wiki_id', 'freebase_movie_id', 'release_date', 'char_name', 'actor_dob', 'actor_gender', 'actor_height', 'actor_ethnicity', 'actor_name', 'actor_age', 'freebase_char_id', 'freebase_actor_id', 'freebase_id_2', 'movie_name']


add for each character the plot of the film
output: char_movie_plots.csv

columns:
wiki_id,freebase_movie_id,release_date,char_name,actor_dob,actor_gender,actor_height,actor_ethnicity,actor_name,actor_age,freebase_char_id,freebase_actor_id,freebase_id_2,movie_name,plot_text

In [ ]:
import pandas as pd
import os

# 1. Caricamento del file delle trame (plot_summaries.txt)
print("Caricamento plot summaries...")
plot_path = 'data/MovieSummaries/plot_summaries.txt'
plot_cols = ['wiki_id', 'plot_text']

df_plots = pd.read_csv(
    plot_path, 
    sep='\t', 
    names=plot_cols, 
    header=None
)

char_movie_path = 'data/processed/char_and_movie.csv'
df_char_movie = pd.read_csv(char_movie_path)

# 3. Esecuzione del MERGE
print("Esecuzione del merge tra personaggi e trame...")
# Usiamo 'wiki_id' come chiave comune
df_final_plots = pd.merge(
    df_char_movie, 
    df_plots, 
    on='wiki_id', 
    how='left'
)

# 4. Verifica dei risultati
print(f"Merge completato. Totale righe: {len(df_final_plots)}")
print(f"Personaggi con trama associata: {df_final_plots['plot_text'].notnull().sum()}")

# Visualizziamo un esempio per Katniss (da Hunger Games)
example = df_final_plots[df_final_plots['char_name'].str.contains("Katniss", na=False)]
if not example.empty:
    display(example[['char_name', 'movie_name', 'plot_text']].head(1))

# 5. Salvataggio del nuovo dataset completo
output_path = 'data/processed/char_movie_plots.csv'
df_final_plots.to_csv(output_path, index=False)
print(f"Dataset salvato in: {output_path}")

Caricamento plot summaries...
Esecuzione del merge tra personaggi e trame...
Merge completato. Totale righe: 450669
Personaggi con trama associata: 308485


,char_name,movie_name,plot_text
363229,Katniss Everdeen,The Hunger Games,The nation of Panem consists of a wealthy Capi...


Dataset salvato in: data/processed/char_movie_plots.csv
